# WGCNA - Timing (3 runs)

💡 **Environment:** `clamp-analyses`  

In [1]:
library(WGCNA)
library(here)

enableWGCNAThreads(nThreads = 2)
set.seed(42)

Loading required package: dynamicTreeCut

Loading required package: fastcluster


Attaching package: ‘fastcluster’


The following object is masked from ‘package:stats’:

    hclust





Attaching package: ‘WGCNA’


The following object is masked from ‘package:stats’:

    cor


here() starts at /home/msubirana/Documents/pivlab/clamp-analyses



Allowing parallel execution with up to 2 working processes.


In [2]:
output_dir <- here("output/model_performance/gtex")
dir.create(output_dir, showWarnings = FALSE, recursive = TRUE)

N_RUNS <- 3

In [3]:
gtex_data <- readRDS(here("output/gtex/df_gtex_fbm_filt.rds"))
datExpr <- as.data.frame(t(gtex_data))

In [ ]:
WGCNA_times <- numeric(N_RUNS)

for (i in 1:N_RUNS) {
  cat("WGCNA run", i, "of", N_RUNS, "\n")
  
  start_time <- Sys.time()
  
  powers <- 1:20
  sft <- pickSoftThreshold(datExpr, powerVector = powers, networkType = "unsigned", verbose = 0)
  soft_power <- sft$fitIndices$Power[which(sft$fitIndices$SFT.R.sq >= 0.9)[1]]
  if (is.na(soft_power)) soft_power <- 7
  
  net <- blockwiseModules(
    datExpr,
    power = soft_power,
    networkType = "unsigned",
    TOMType = "unsigned",
    minModuleSize = 30,
    mergeCutHeight = 0.25,
    numericLabels = TRUE,
    verbose = 3,
    maxBlockSize = ncol(datExpr),
    nThreads = 2
  )
  
  end_time <- Sys.time()
  WGCNA_times[i] <- as.numeric(difftime(end_time, start_time, units = "mins"))
  cat("Run", i, "time:", WGCNA_times[i], "minutes\n\n")
}

WGCNA run 1 of 3 
   Power SFT.R.sq  slope truncated.R.sq mean.k. median.k. max.k.
1      1   0.5170  1.470          0.964 6160.00  6160.000  10200
2      2   0.0719 -0.259          0.801 2560.00  2330.000   6010
3      3   0.5500 -0.941          0.837 1270.00  1020.000   3960
4      4   0.6950 -1.260          0.873  705.00   498.000   2770
5      5   0.7430 -1.430          0.889  421.00   261.000   2020
6      6   0.7740 -1.530          0.913  266.00   145.000   1520
7      7   0.7820 -1.620          0.917  175.00    85.900   1170
8      8   0.7840 -1.690          0.924  120.00    53.300    923
9      9   0.7910 -1.730          0.935   83.90    33.800    737
10    10   0.8060 -1.750          0.948   60.20    21.800    595
11    11   0.8180 -1.770          0.959   44.10    14.400    486
12    12   0.8320 -1.770          0.969   32.80     9.610    400
13    13   0.8400 -1.790          0.973   24.80     6.550    332
14    14   0.8500 -1.790          0.979   19.00     4.520    278
15    1

In [ ]:
WGCNA_time_minutes <- WGCNA_times
names(WGCNA_time_minutes) <- paste0("run", 1:N_RUNS)
saveRDS(WGCNA_time_minutes, file.path(output_dir, "WGCNA_time_minutes.rds"))
cat("WGCNA times:", WGCNA_time_minutes, "minutes\n")